# QC confound 02 — Does PBS composition survive adjustment for technical covariates?

With PBS one-hot features, a cell-level model can only learn one ER+ probability per
subtype, so its prediction for a specimen is fully determined by the specimen's PBS
composition (fractions of pbs-1/2/3 among labelled noise cells). The question can
therefore be asked at the **specimen level**, where the sample size is honest (number
of specimens, not cells) and adjusting for specimen-level covariates is well defined.

Covariates were fixed before seeing results (`PRESPECIFIED_TECHNICAL_COVARIATES`):
- `median_log1p_total_counts_retained` — sequencing depth
- `median_complexity_retained` — library complexity (log genes / log counts)
- `median_pct_counts_mt_all` — dissociation-stress proxy
- `qc_genes_lower` — Pal's per-sample gene threshold

Questions:
1. Does PBS composition track technical covariates **within** each condition? If so,
   technical variation moves PBS composition regardless of biology.
2. Does PBS composition still predict condition after residualizing on the covariates
   (fit inside each training fold)?
3. Does PBS add anything to a technical-only model?

**Power caveat.** There are about 20 specimens. A result that survives adjustment is
meaningful evidence against the confound. A result that disappears after adjustment
is weaker evidence for it, because adjustment also removes any biology correlated with
the covariates.

In [ ]:
import logging

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from signals_in_the_noise.analysis.noise_phenotypes import ISCB_SCS_PBS_THRESHOLDS, PBS_LABELS
from signals_in_the_noise.config import get_data_path
from signals_in_the_noise.preprocessing.gse161529 import GSE161529
from signals_in_the_noise.preprocessing.specimens import Cohort
from signals_in_the_noise.utils.logging_config import setup_logging

setup_logging(logging.INFO)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

POSITIVE, NEGATIVE = "ER+ tumour", "Normal"
SEED = 0
OUTPUT_DIR = get_data_path("processed") / "qc_confound"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
gse = GSE161529()
cohort = Cohort.from_objects(gse.objects.values(), [POSITIVE, NEGATIVE])
cohort.overview().groupby("condition").agg(specimens=("n_cells", "size"), cells=("n_cells", "sum"))

In [ ]:
from signals_in_the_noise.modeling.datasets import build_cell_table, standard_arms
from signals_in_the_noise.modeling.experiments import PUBLISHED_PROTOCOL, reference_specimens

reference_arm = standard_arms(ISCB_SCS_PBS_THRESHOLDS, names=["noise_pbs"])[0]
reference_table = build_cell_table(
    cohort.obs, cohort.conditions, reference_arm, patients=cohort.patients
)
included = reference_specimens(reference_table, PUBLISHED_PROTOCOL["min_reference_cells"])
min_cells = PUBLISHED_PROTOCOL["min_reference_cells"]
print(f"{len(included)} specimens have >= {min_cells} PBS-labelled noise cells")

In [ ]:
from signals_in_the_noise.analysis.confounders import (
    PRESPECIFIED_TECHNICAL_COVARIATES,
    feature_covariate_correlations,
    specimen_covariates,
)
from signals_in_the_noise.modeling.datasets import specimen_composition

PBS = list(PBS_LABELS)
TECHNICAL = list(PRESPECIFIED_TECHNICAL_COVARIATES)

composition = specimen_composition(
    reference_table.loc[reference_table["specimen_id"].isin(included)]
)
covariates = specimen_covariates({sid: cohort.specimens[sid] for sid in included})
table = composition.join(covariates.drop(columns=["condition", "patient_id"]))
table[["condition", "patient_id", *PBS, *TECHNICAL, "noise_fraction"]]

## 1. Does PBS composition track technical covariates?

Spearman ρ between each PBS fraction and each covariate, over all included specimens
and within each condition. Strong within-condition correlations mean the covariate
moves PBS composition independently of condition.

In [ ]:
correlations = feature_covariate_correlations(table, PBS, [*TECHNICAL, "noise_fraction"])
groups = correlations["group"].unique()
fig, axes = plt.subplots(1, len(groups), figsize=(5 * len(groups), 3.5), sharey=True)
for ax, group in zip(axes, groups):
    grid = correlations[correlations["group"] == group].pivot(
        index="covariate", columns="feature", values="rho"
    )
    sns.heatmap(
        grid, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax, cbar=False
    )
    n = int(correlations.loc[correlations["group"] == group, "n"].max())
    ax.set_title(f"{group} (n={n})")
plt.tight_layout()
plt.show()
correlations.round(3)

## 2. Nested and adjusted specimen-level models

`PBS | X` residualizes the PBS fractions on covariates X inside each training fold,
then classifies on the residuals. Patients are held out; the permutation null shuffles
condition labels across specimens.

In [ ]:
from signals_in_the_noise.modeling.evaluation import logistic_regression_factory
from signals_in_the_noise.modeling.experiments import compare_feature_sets

N_PERMUTATIONS = 1000
feature_sets = {
    "technical only": (TECHNICAL, None),
    "PBS only": (PBS, None),
    "PBS + technical": (PBS + TECHNICAL, None),
    "PBS | technical": (PBS, TECHNICAL),
}
feature_sets |= {f"PBS | {c}": (PBS, [c]) for c in [*TECHNICAL, "noise_fraction"]}

adjusted = compare_feature_sets(
    table,
    feature_sets,
    positive_label=POSITIVE,
    n_permutations=N_PERMUTATIONS,
    model_factory=logistic_regression_factory(C=1.0),
    group="patient_id",
    seed=SEED,
)
adjusted.round(3)

In [ ]:
table.to_csv(OUTPUT_DIR / "02_specimen_table.csv")
correlations.to_csv(OUTPUT_DIR / "02_pbs_covariate_correlations.csv", index=False)
adjusted.to_csv(OUTPUT_DIR / "02_adjusted_models.csv")

## Reading the results

| Observation | Meaning |
|---|---|
| `PBS only` significant, `PBS | technical` still clearly above its null | PBS composition carries condition information beyond the named technical factors. |
| `PBS | X` collapses for one specific covariate X | That covariate is the likely driver; check its row in notebook 00. |
| `technical only` ≥ `PBS only`, and `PBS + technical` ≈ `technical only` | PBS adds nothing beyond sample-level technical covariates. |
| Strong within-condition ρ between a PBS fraction and depth/complexity | Technical variation moves PBS composition directly. |